## Writing a service unit (and drop-ins)

A `.service` file is INI-style — sections in `[brackets]`, `key=value` lines. A complete unit to run a script:

```ini
# /etc/systemd/system/myapp.service
[Unit]
Description=My Application
After=network-online.target

[Service]
Type=simple
ExecStart=/usr/local/bin/myapp
Restart=on-failure
User=myapp

[Install]
WantedBy=multi-user.target
```

Three sections: **`[Unit]`** — description and ordering (**`After=`** = "don't start me until this is up"). **`[Service]`** — how to run it: **`Type=simple`** (the default; `ExecStart` *is* the main process), **`ExecStart=`** an **absolute path**, **`Restart=on-failure`** (the killer feature — auto-restart on crash), and **`User=`** to drop privileges. **`[Install]`** — **`WantedBy=multi-user.target`** means "start me at boot" once enabled.

Activate it — remembering the golden rule:

```bash
sudo systemctl daemon-reload        # read the new file
sudo systemctl enable --now myapp   # start now + at boot
journalctl -u myapp -f              # watch its logs
```

**Drop-ins — tweak a *vendor* unit without editing it.** Editing `/lib`'s file gets overwritten on update; instead **`systemctl edit nginx`** creates a small override in `/etc/systemd/system/nginx.service.d/override.conf`, merged over the vendor unit:

```ini
[Service]
LimitNOFILE=65536
```

It auto-runs `daemon-reload` (you still `restart`). `systemctl cat nginx` shows the merged result; `systemctl revert nginx` removes the override. **Prefer drop-ins** — they survive updates and are trivial to undo.
